In [25]:
from src.gillespie.simulation_config import SimulationConfig

In [26]:
config= SimulationConfig()


In [27]:
print(config.cell_parameters["exhausted"].default_id)

(-2,)


In [28]:
print(config.cell_parameters["base"].K)

4000


In [29]:
from src.gillespie.clone_factory import CloneFactory
factory = CloneFactory(config= config)
test_clone = factory.create_clone(clone_id=(0,),clone_type= "mutated")
test_clone2 = factory.create_clone(clone_id=(1,), clone_type= "mutated", N=4)
test_clone3 = factory.create_clone(clone_id=(3,), clone_type= "mutated", N= None)

In [30]:
print(test_clone.N)
print(test_clone2.N)
print(test_clone3.N)

0
4
0


In [31]:
test4 = factory.create_clone(clone_type="mutated")
print(test4.cell_parameters)

CellTypeConfig(default_id=(-3,), N=0, K=8000, lambda0=0.006, mu=0.002, nu=0.0, omega_exhaust=0.0, fitness_gain=0.0, next_mutation='')


In [32]:
from src.gillespie.tumor_simulation import TumorSimulation



In [33]:
simulation = TumorSimulation(config=config)
for cloneid, clones in simulation.tissue_state.clones.items():  
    print(cloneid, clones.clone_id)

arrancando con parametros:
SimulationConfig(OMEGA=4000, cell_parameters={'base': CellTypeConfig(default_id=(), N=4000, K=4000, lambda0=0.005, mu=0.002, nu=0.0, omega_exhaust=0.0, fitness_gain=0.0, next_mutation='mutated'), 'immune': CellTypeConfig(default_id=(-1,), N=0, K=2000, lambda0=0.005, mu=0.0, nu=0.0, omega_exhaust=7.5e-07, fitness_gain=0.0, next_mutation=''), 'mutated': CellTypeConfig(default_id=(-3,), N=0, K=8000, lambda0=0.006, mu=0.002, nu=0.0, omega_exhaust=0.0, fitness_gain=0.0, next_mutation=''), 'exhausted': CellTypeConfig(default_id=(-2,), N=0, K=None, lambda0=0.0, mu=0.002, nu=0.0, omega_exhaust=0.0, fitness_gain=0.0, next_mutation='')}, T_max=200, seed=None, decline=0.0, Kmin=1, theta_I=1.25e-07, beta=1.0000000000000001e-07, d1_0=0.0, d2_0=0.0, instability_0=0.0, buildup_0=0.0, base_instability_buildup=0.0, mutation_instability_jump=0.0, mutation_buildup_gain=0.0, verbose=True, scale=True, decay=False, use_logistic=True, use_logistic_adapted=True, crowding_strategy=<s

In [34]:
import pandas as pd

simulation.tissue_state.snapshot()

{(): {'Type': 'base',
  'N': 4000,
  't': 0.0,
  'rb': 0.005,
  'rd': 0.002,
  'rm': 0.0,
  're': 0.0,
  'instability': 0.0,
  'buildup': 0.0,
  'K': 4000},
 (-3,): {'Type': 'mutated',
  'N': 0,
  't': 0.0,
  'rb': 0.006,
  'rd': 0.002,
  'rm': 0.0,
  're': 0.0,
  'instability': 0.0,
  'buildup': 0.0,
  'K': 8000},
 (-1,): {'Type': 'immune',
  'N': 0,
  't': 0.0,
  'rb': 0.005,
  'rd': 0.0,
  'rm': 0.0,
  're': 7.5e-07,
  'instability': 0.0,
  'buildup': 0.0,
  'K': 2000},
 (-2,): {'Type': 'exhausted',
  'N': 0,
  't': 0.0,
  'rb': 0.0,
  'rd': 0.002,
  'rm': 0.0,
  're': 0.0,
  'instability': 0.0,
  'buildup': 0.0,
  'K': None}}

In [35]:

# simulation.step()

# simulation.run()
# simulation.step()
# simulation.run()

# for i in range(2):
#     try:
#         # Intentamos recuperar el clon y meter la mutación
#         target_clone = simulation.tissue_state.clones[()]
#         simulation._introduce_mutation(target_clone)
#         print(f"Mutación introducida con éxito en la iteración {i}")
        
#     except AssertionError as e:
#         # Si el assert de tumor_simulation.py falla, atrapamos el error aquí
#         print(f"Iteración {i} saltada: {e}")
#         continue

In [36]:
simulation.tissue_state.print_pop_map()

clone_type | count
----------+------
base      | 4000
exhausted | 0
immune    | 0
mutated   | 0


In [37]:
simulation.step()
simulation.events

[Event(kind=<EventType.DEATH: 'death'>, clone_id=(), rate=8.0, clone_type='base', reaction_number=0)]

In [38]:
simulation.step(return_matrix=True)
    
     


0.03147409771023978
Kind          | Type        | Clone ID        | Rate       | N              
----------------------------------------------------------------------------
BIRTH        | base         | ()              | 8.0016    | 3999
DEATH        | base         | ()              | 7.9980    | 3999
----------------------------------------------------------------------------


True

In [39]:
beta = config.beta
mut=  simulation.tissue_state.pop_map.get("mutated")
imm = simulation.tissue_state.pop_map.get("immune")
print(mut)


0


In [40]:

print ( beta *mut*imm)

0.0


In [41]:

obj_imm = simulation.tissue_state.clones[()]
base = obj_imm.birth_rate * simulation.tissue_state.pop_map.get(obj_imm.get_type(),0) 
crowding_effect = obj_imm.config.crowding_strategy.crowding(obj_imm,tissue_state=simulation.tissue_state)
print(obj_imm.crowding_numerator(simulation.tissue_state))#


4000


In [42]:

print(obj_imm.actual_K)


6667


In [43]:

print(base)


20.0


In [44]:

print(crowding_effect)


0.40002999850007503


In [45]:
#TODO: Improve efficiency: right now every clone has its own id, which is fine but we need to store the calculation of the rates inside the subclass instead of running it for each clone. (calculate once the rates at each time and pass it to all subclasses )
simulation.tissue_state.print_pop_map()
for clones in simulation.tissue_state.clones.values():
    print("--------------------------------")
    print(clones)
    print(clones.clone_id)
    print(clones.cell_parameters)
    print(clones.actual_K)
    print(clones.birth_rate_effective(simulation.tissue_state))
    

clone_type | count
----------+------
base      | 4000
exhausted | 0
immune    | 0
mutated   | 0
--------------------------------
base
()
CellTypeConfig(default_id=(), N=4000, K=4000, lambda0=0.005, mu=0.002, nu=0.0, omega_exhaust=0.0, fitness_gain=0.0, next_mutation='mutated')
6667
8.0005999700015
--------------------------------
mutated
(-3,)
CellTypeConfig(default_id=(-3,), N=0, K=8000, lambda0=0.006, mu=0.002, nu=0.0, omega_exhaust=0.0, fitness_gain=0.0, next_mutation='')
12000
0.0
--------------------------------
immune
(-1,)
CellTypeConfig(default_id=(-1,), N=0, K=2000, lambda0=0.005, mu=0.0, nu=0.0, omega_exhaust=7.5e-07, fitness_gain=0.0, next_mutation='')
2000
0.0
--------------------------------
exhausted
(-2,)
CellTypeConfig(default_id=(-2,), N=0, K=None, lambda0=0.0, mu=0.002, nu=0.0, omega_exhaust=0.0, fitness_gain=0.0, next_mutation='')
inf
0.0


In [46]:
simulation.tissue_state.clones[()].crowding_numerator(simulation.tissue_state)

4000

In [47]:
simulation.tissue_state.clones[()].N

4000

In [48]:
simulation.step()

True